<a href="https://colab.research.google.com/github/conluoi123/vnfood_vision/blob/feat_crawler/notebooks/00_crawler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
=============================================================================
CRAWL ẢNH + CÔNG THỨC TỪ COOKPAD VN
=============================================================================
Nguồn duy nhất: https://cookpad.com/vn
- Ảnh lấy thẳng từ trang công thức (img-global.cpcdn.com) → ảnh thực tế món ăn
- Công thức: nguyên liệu + cách làm
"""

# ============================================================
# 0. CÀI THƯ VIỆN
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Kiểm tra và cài thư viện...")
for pkg in ["requests", "beautifulsoup4", "Pillow", "lxml"]:
    pip_install(pkg)
print("Thư viện sẵn sàng\n")

# ============================================================
# 1. IMPORT
# ============================================================
import os, re, json, time, random, logging, hashlib, urllib.parse
from io import BytesIO
from pathlib import Path
from datetime import datetime

import requests
from bs4 import BeautifulSoup
from PIL import Image
import csv
# ============================================================
# 2. GOOGLE DRIVE & THƯ MỤC
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/crawled')
    print("Google Drive đã kết nối")
except Exception:
    ROOT = Path('./crawled')
    print(f"Local mode → {ROOT.resolve()}")

IMG_DIR    = ROOT / 'dataset' / 'raw_images'
RECIPE_DIR = ROOT / 'recipes'
LOG_DIR    = ROOT / 'logs'
for d in [IMG_DIR, RECIPE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Lưu tại: {ROOT}\n")

# ============================================================
# 3. LOGGING
# ============================================================
_logfile = LOG_DIR / f'cookpad_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(_logfile, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('cookpad_crawler')

# ============================================================
# 4. CẤU HÌNH
# ============================================================

# Số ảnh mục tiêu mỗi món
TARGET_IMAGES = 250

# Số công thức tối đa thu thập mỗi món
TARGET_RECIPES = 10

IMG_SIZE = (224, 224)   # Chuẩn cho hầu hết CNN

# Base URL tìm kiếm Cookpad VN
SEARCH_URL = "https://cookpad.com/vn/tim-kiem/{query}"

# Danh sách món: key = label (tên thư mục / nhãn khi train)
FOOD_LIST = {
    "pho_bo"       : "Phở bò",
    "banh_mi_thit" : "Bánh mì thịt",
    "banh_xeo"     : "Bánh xèo",
    "goi_cuon"     : "Gỏi cuốn",
    "bun_bo_hue"   : "Bún bò Huế",
    "com_tam"      : "Cơm tấm",
    "banh_cuon"    : "Bánh cuốn",
    "bun_cha"      : "Bún chả",
    "mi_quang"     : "Mì Quảng",
    "hu_tieu"      : "Hủ tiếu",
    "bun_rieu"     : "Bún riêu",
    "bun_thit_nuong": "Bún thịt nướng",
    "com_ga"       : "Cơm gà",
    "com_chien"    : "Cơm chiên",
    "chao"         : "Cháo",
    "sup_cua"      : "Súp cua",
    "bun_dau_mam_tom": "Bún đậu mắm tôm",
    "banh_bao"     : "Bánh bao",
    "banh_gio"     : "Bánh giò",
    "banh_chung"   : "Bánh chưng",
    "banh_tet"     : "Bánh tét",
    "nem_ran"      : "Nem rán",
    "ga_ran"       : "Gà rán",
    "ga_nuong"     : "Gà nướng",
    "banh_canh"    : "Bánh canh",
    "banh_khot"    : "Bánh khọt",
    "banh_can"     : "Bánh căn",
    "banh_bot_loc" : "Bánh bột lọc",
    "banh_gio"     : "Bánh giò",
    "ca_kho_to"    : "Cá kho tộ",
    "banh_duc"     : "Bánh đúc",
    "banh_beo"     : "Bánh bèo",
    "cao_lau"      : "Cao lầu",
    "nem_chua"     : "Nem chua",
    "banh_pia"     : "Bánh pía",
    "lau_thai"     : "Lẩu Thái",
    "lau_hai_san"  : "Lẩu hải sản",
    "canh_chua"    : "Canh chua",
    "bun_mam"      : "Bún mắm",
    "banh_trang_tron": "Bánh tráng trộn",
    "banh_trang_nuong": "Bánh tráng nướng",
    "xoai_lac"     : "Xoài lắc",
    "tra_sua"      : "Trà sữa",
}

# Headers pool — xoay vòng để tránh block
_HEADERS = [
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept-Language": "vi-VN,vi;q=0.9,en;q=0.7",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Referer": "https://www.google.com/",
    },
    {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15",
        "Accept-Language": "vi-VN,vi;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://cookpad.com/vn",
    },
    {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        "Accept-Language": "vi,en-US;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://cookpad.com/vn",
    },
]

def _h():
    return random.choice(_HEADERS)

def _sleep(lo=1.5, hi=3.5):
    time.sleep(random.uniform(lo, hi))

# ============================================================
# 5. HTTP SESSION với retry tự động
# ============================================================
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def make_session():
    s = requests.Session()
    retry = Retry(
        total=2,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://",  HTTPAdapter(max_retries=retry))
    return s

SESSION = make_session()

# ============================================================
# 6. LẤY DANH SÁCH LINK
# ============================================================
def get_recipe_links(food_name: str, max_pages: int = 5) -> list[str]:
    seen  = set()
    links = []

    for page in range(1, max_pages + 1):
        url = SEARCH_URL.format(query=urllib.parse.quote(food_name))
        if page > 1:
            url += f"?page={page}"

        log.info(f"    Trang {page}: {url}")
        try:
            r = SESSION.get(url, headers=_h(), timeout=10)

            # Nếu hết trang (404) thì dừng luôn
            if r.status_code == 404:
                log.info(f"    Hết trang tại page {page}")
                break

            r.raise_for_status()

        except Exception:
            log.debug(f"    Bỏ qua lỗi trang {page}")
            break

        soup = BeautifulSoup(r.text, 'lxml')

        found_this_page = 0
        for a in soup.select('a[href^="/vn/cong-thuc/"]'):
            href = a['href'].split('?')[0]
            if "tao-moi" in href or "chinh-sua" in href:
                continue
            if re.match(r'^/vn/cong-thuc/\d+', href) and href not in seen:
                seen.add(href)
                links.append(href)
                found_this_page += 1

        log.info(f"    → {found_this_page} link mới (tổng: {len(links)})")

        if found_this_page == 0:
            break

        _sleep(1.5, 3.0)

    return links

# ============================================================
# 7. LẤY CHI TIẾT CÔNG THỨC
# ============================================================
_IMG_CDN_RE = re.compile(r'https://img-global\.cpcdn\.com/recipes/[^"\')\s]+')

def get_recipe_detail(path: str) -> dict | None:
    url = "https://cookpad.com" + path
    try:
        r = SESSION.get(url, headers=_h(), timeout=10)
        r.raise_for_status()
    except Exception as e:
        log.warning(f"    Lỗi tải {path}: {e}")
        return None

    soup = BeautifulSoup(r.text, 'lxml')

    h1 = soup.find('h1')
    title = h1.get_text(strip=True) if h1 else "Không rõ"

    author_el = soup.select_one('[itemprop="author"]') or soup.select_one('a[href*="/nguoi-su-dung/"]')
    author = author_el.get_text(strip=True) if author_el else ""

    nguyen_lieu = []
    for el in soup.select('[itemprop="recipeIngredient"]'):
        t = el.get_text(strip=True)
        if t and t not in nguyen_lieu:
            nguyen_lieu.append(t)

    if not nguyen_lieu:
        for el in soup.select('[class*="ingredient"] li, [id*="ingredient"] li'):
            t = el.get_text(strip=True)
            if t and t not in nguyen_lieu:
                nguyen_lieu.append(t)

    cach_lam = []
    for el in soup.select('[itemprop="recipeInstructions"]'):
        t = el.get_text(strip=True)
        if t and t not in cach_lam:
            cach_lam.append(t)

    if not cach_lam:
        for el in soup.select('[class*="step"] p, [id*="step"] p, ol.steps li'):
            t = el.get_text(strip=True)
            if t and t not in cach_lam:
                cach_lam.append(t)

    title_lower = title.lower()
    tu_khoa_nhieu = ["chay", "xào", "trộn", "áp chảo", "chiên", "thập cẩm"]
    if any(tu in title_lower for tu in tu_khoa_nhieu):
        log.debug(f"    Bỏ qua món không chuẩn: {title}")
        return None

    img_url = None
    meta_img = soup.find('meta', property='og:image')
    if meta_img and 'cpcdn.com' in meta_img.get('content', ''):
        img_url = meta_img['content']

    if not img_url:
        for img_tag in soup.find_all('img', src=True):
            src = img_tag['src']
            if 'cpcdn.com/recipes' in src:
                img_url = src
                break

    if not img_url:
        m = _IMG_CDN_RE.search(r.text)
        if m:
            img_url = m.group(0).split('"')[0]

    if not nguyen_lieu and not cach_lam:
        log.debug(f"    Bỏ qua (không có nội dung): {path}")
        return None

    return {
        "url"        : url,
        "tieu_de"    : title,
        "tac_gia"    : author,
        "nguyen_lieu": nguyen_lieu,
        "cach_lam"   : cach_lam,
        "img_url"    : img_url,
    }

# ============================================================
# 8. XỬ LÝ & LƯU ẢNH
# ============================================================
def is_valid_image(data: bytes, min_kb: int = 8) -> bool:
    if len(data) < min_kb * 1024:
        return False
    try:
        img = Image.open(BytesIO(data))
        img.verify()
        img = Image.open(BytesIO(data))
        w, h = img.size
        return w >= 100 and h >= 100 and img.format in ('JPEG', 'PNG', 'WEBP')
    except Exception:
        return False

def save_image_224(data: bytes, path: Path) -> bool:
    try:
        img = Image.open(BytesIO(data)).convert('RGB')
        w, h = img.size
        s = min(w, h)
        img = img.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2))
        # Image.LANCZOS thay thế cho Image.ANTIALIAS ở các bản Pillow mới
        img = img.resize(IMG_SIZE, Image.LANCZOS)
        img.save(path, 'JPEG', quality=92)
        return True
    except Exception as e:
        log.debug(f"    save_image lỗi: {e}")
        return False

def download_image(img_url: str, save_dir: Path, existing: set) -> bool:
    if not img_url:
        return False

    big_url = re.sub(r'/\d+x\d+[^/]*/', '/640x640cq80/', img_url)
    url_hash = hashlib.md5(img_url.encode()).hexdigest()[:12]
    if url_hash in existing:
        return False

    for try_url in [big_url, img_url]:
        try:
            r = SESSION.get(try_url, headers=_h(), timeout=15)
            if r.status_code == 200 and is_valid_image(r.content):
                dst = save_dir / f"{url_hash}.jpg"
                if save_image_224(r.content, dst):
                    existing.add(url_hash)
                    return True
        except Exception:
            continue
    return False

# ============================================================
# 9. CACHE CÔNG THỨC
# ============================================================
CSV_FILE = RECIPE_DIR / 'recipes.csv'

def init_csv():
    if not CSV_FILE.exists():
        with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                "label",
                "ten_mon",
                "url",
                "tieu_de",
                "tac_gia",
                "nguyen_lieu",
                "cach_lam"
            ])

def append_recipe_to_csv(label, food_name, detail):
    CSV_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(CSV_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)

        writer.writerow([
            label,
            food_name,
            detail['url'],
            detail['tieu_de'],
            detail['tac_gia'],
            " | ".join(detail['nguyen_lieu']),
            " | ".join(detail['cach_lam'])
        ])

def load_existing_urls():
    if not CSV_FILE.exists():
        return set()

    urls = set()
    with open(CSV_FILE, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            urls.add(row['url'])
    return urls
# ============================================================
# 10. CRAWL 1 MÓN
# ============================================================
def crawl_one_food(label: str, food_name: str):
    save_dir = IMG_DIR / label
    save_dir.mkdir(exist_ok=True)

    existing = {f.stem for f in save_dir.glob('*.jpg')}
    img_count = len(existing)

    global existing_urls

    log.info(f"  Hiện có: {img_count} ảnh")

    if img_count >= TARGET_IMAGES:
        log.info("Đã đủ ảnh — bỏ qua")
        return

    pages_needed = max(2, (TARGET_IMAGES // 8) + 1)
    all_links = get_recipe_links(food_name, max_pages=pages_needed)
    log.info(f"  Tổng link tìm được: {len(all_links)}")

    for i, link in enumerate(all_links):
        if img_count >= TARGET_IMAGES:
            break

        log.info(f"  [{i+1}/{len(all_links)}] {link}")
        detail = get_recipe_detail(link)

        if detail is None:
            _sleep(1, 2)
            continue

        # Lưu CSV
        if detail['url'] not in existing_urls:
            append_recipe_to_csv(label, food_name, detail)
            existing_urls.add(detail['url'])
            log.info(f"CSV: {detail['tieu_de']}")

        # Lưu ảnh
        if detail.get('img_url'):
            if download_image(detail['img_url'], save_dir, existing):
                img_count += 1
                if img_count % 10 == 0:
                    log.info(f"       📸 Ảnh: {img_count}/{TARGET_IMAGES}")

        _sleep(0.5, 1.2)

    log.info(f"  Kết quả: {img_count} ảnh")
# ============================================================
# 11. THỐNG KÊ
# ============================================================
def print_summary():
    print("\n" + "="*60)
    print("  THỐNG KÊ DATASET COOKPAD (CSV)")
    print("="*60)
    print(f"  {'Món ăn':<22} {'Ảnh':>6}  {'CT':>4}")
    print("-"*60)

    # Đếm công thức từ CSV
    recipe_count = {}
    if CSV_FILE.exists():
        with open(CSV_FILE, encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                label = row['label']
                recipe_count[label] = recipe_count.get(label, 0) + 1

    total_img = 0

    for label, name in FOOD_LIST.items():
        folder = IMG_DIR / label
        n_img = len(list(folder.glob('*.jpg'))) if folder.exists() else 0
        n_ct  = recipe_count.get(label, 0)

        total_img += n_img

        print(f"  {name:<22} {n_img:>6}  {n_ct:>4}")

    print("-"*60)
    print(f"  {'TỔNG':<22} {total_img:>6}")
    print("="*60)

# ============================================================
# 12. MAIN
# ============================================================
def main():
    log.info("=" * 55)
    log.info("COOKPAD VN CRAWLER — CSV MODE")
    log.info(f"Mục tiêu: {TARGET_IMAGES} ảnh / món")
    log.info("=" * 55)

    # init CSV + load URL
    init_csv()

    global existing_urls
    existing_urls = load_existing_urls()

    for label, food_name in FOOD_LIST.items():
        print(f"\n{'='*55}")
        print(f"  {food_name.upper()}  [{label}]")
        print(f"{'='*55}")
        crawl_one_food(label, food_name)
        _sleep(4, 8)

    print_summary()
    print("\n HOÀN TẤT!")

if __name__ == '__main__':
    main()

Kiểm tra và cài thư viện...
Thư viện sẵn sàng

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive đã kết nối
Lưu tại: /content/drive/MyDrive/crawled


  PHỞ BÒ  [pho_bo]



  BÁNH MÌ THỊT  [banh_mi_thit]

  BÁNH XÈO  [banh_xeo]

  GỎI CUỐN  [goi_cuon]



  BÚN BÒ HUẾ  [bun_bo_hue]



  CƠM TẤM  [com_tam]

  BÁNH CUỐN  [banh_cuon]

  BÚN CHẢ  [bun_cha]

  MÌ QUẢNG  [mi_quang]

  HỦ TIẾU  [hu_tieu]

  BÚN RIÊU  [bun_rieu]

  BÚN THỊT NƯỚNG  [bun_thit_nuong]

  CƠM GÀ  [com_ga]

  CƠM CHIÊN  [com_chien]

  CHÁO  [chao]

  SÚP CUA  [sup_cua]

  BÚN ĐẬU MẮM TÔM  [bun_dau_mam_tom]

  BÁNH BAO  [banh_bao]

  BÁNH GIÒ  [banh_gio]

  BÁNH CHƯNG  [banh_chung]

  BÁNH TÉT  [banh_tet]

  NEM RÁN  [nem_ran]

  GÀ RÁN  [ga_ran]

  GÀ NƯỚNG  [ga_nuong]

  BÁNH CANH  [banh_canh]

  BÁNH KHỌT  [banh_khot]

  BÁNH CĂN  [banh_can]

  BÁNH BỘT LỌC  [banh_bot_loc]

  CÁ KHO TỘ  [ca_kho_to]

  BÁNH ĐÚC  [banh_duc]

  BÁNH BÈO  [banh_beo]

  CAO LẦU  [cao_lau]

  NEM CHUA  [nem_chua]

  BÁNH PÍA  [banh_pia]

  LẨU THÁI  [lau_thai]

  LẨU HẢI SẢN  [lau_hai_san]

  CANH CHUA  [canh_chua]

  BÚN MẮM  [bun_mam]

  BÁNH TRÁNG TRỘN  [banh_trang_tron]

  BÁNH TRÁNG NƯỚNG  [banh_trang_nuong]

  XOÀI LẮC  [xoai_lac]

  TRÀ SỮA  [tra_sua]

  THỐNG KÊ DATASET COOKPAD (CSV)
  Món ăn                